# Stage 11 -- Final ICD-10 Decision + Evaluation

**Input**:
- `stage_06b_history_context/history_codes.json` -- codes carried in from prior admissions
- `stage_06c_icd_mapping/icd_candidates.json` -- codes mapped from this visit's inferred diagnoses

**Output**: `stage_07_final_decision/final_icd_decision.json` per admission, plus
`patient_records/stage_07_summary.json`

This is the first stage that actually **decides**. Everything upstream deliberately deferred
judgement: 6b carries every prior code forward, 6c keeps every plausible mapping. Here they are
combined, scored, thresholded, and scored against `ground_truth.txt`.

It is also the pipeline's **measurement harness** -- the first point with a real objective
function, so it reports an ablation (history alone / symptoms alone / combined) that shows what
each half actually contributes as the upstream stages change.

## What the measurements say, before any of the code below

Measured across all 15 admissions (247 ground-truth codes):

| Decision rule | macro F1 | micro F1 | Precision | Recall | preds |
|---|---|---|---|---|---|
| 6b all codes (no decision) | 0.363 | 0.367 | 0.297 | 0.482 | 401 |
| **6b confidence >= 0.3** | **0.379** | **0.396** | 0.354 | 0.449 | 314 |
| 6b >= 0.3 + 6c (well-supported only) | 0.378 | 0.394 | 0.351 | 0.449 | 316 |
| 6b >= 0.3 + all 6c | 0.360 | 0.386 | 0.322 | 0.482 | 370 |
| union(6b, 6c), no threshold | 0.346 | 0.361 | 0.278 | 0.514 | 457 |
| 6c alone | 0.032 | 0.058 | 0.145 | 0.036 | 62 |

**Thresholding history is what improves the result; adding the symptom side currently does not.**
A light threshold on 6b (0.3) lifts F1 from 0.363 to 0.379 by dropping ~90 low-confidence priors
that were mostly wrong. Every configuration that folds in 6c lands at or below that.

## The agreement signal does not work, and that overturns an earlier assumption

The design bet was that a code supported by *both* history and the current note would be the
strongest evidence available -- 6d deliberately withholds prior codes from its prompt so that
agreement would be independent and therefore meaningful. Measured:

| Source | Correct | Rate |
|---|---|---|
| In **both** 6b and 6c | 1/6 | **16.7%** |
| 6b only | 118/395 | 29.9% |
| 6c only | 8/56 | 14.3% |

Codes appearing in both are *less* likely to be correct than history-only codes -- and there are
only **6 of them in the entire dataset**. The two sources barely overlap, so agreement is too
rare to build a decision rule on. The independence was worth engineering (it means the overlap
is a real signal rather than an artifact), but the overlap itself is empty.

At n=6, this is not a statistically firm conclusion -- it is a statement that the expected signal
did not materialise at this scale, not proof it never will.

## Why the symptom side contributes so little, and what would change it

6c produced 62 candidates against 247 ground-truth codes, so even perfect precision caps its
recall near 25%. Two diagnosable causes, neither fundamental:

1. **Coverage** -- most ground-truth codes are chronic comorbidities and Z-codes (17.4% of the
   ground truth is chapter Z: homelessness, palliative care encounters, long-term drug use).
   Nothing in a current-visit symptom implies those. History covers them; the note does not
   restate them.
2. **Specificity** -- 6c's category-level recall (8.9%) is more than double its exact recall
   (3.6%), so over half of its near-hits die on the final digit: `N179` unspecified vs `N170`
   *with tubular necrosis*. Recovering that needs labs and imaging detail that a diagnosis name
   does not carry.

So the honest reading is that the symptom chain does not yet earn a place in the final
prediction **on this cohort at this scale** -- and the ablation below is what will show it
starting to, as those two causes are addressed.

## Pipeline position

```
Stage 5   Ontology Routing Agent      symptoms  -> SNOMED concepts
Stage 6   Cross-Symptom Routing       concepts  -> clusters
Stage 7   Diagnosis Inference         clusters  -> named diagnoses
Stage 8   ICD-10 Mapping              diagnoses -> ICD codes
Stage 9   Prior-Admission Codes       history   -> ICD codes      (independent)
Stage 10  Lab/Vital Rules             labs      -> ICD codes      (independent)
Stage 11  Final Decision              combines 9 + 8 + 10
```
Run top to bottom. Stages 9 and 10 read only their own inputs, so they may run at any point
before Stage 11.


## 1. Setup

In [1]:
import json
import re
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if (ROOT / "pipeline.py").exists():
    NB_DIR = ROOT
elif (ROOT / "notebooks" / "pipeline.py").exists():
    NB_DIR = ROOT / "notebooks"
else:
    NB_DIR = ROOT.parent / "notebooks"

PROJECT_ROOT    = NB_DIR.parent
RECORDS_DIR     = PROJECT_ROOT / "patient_records"
STAGE_6B_OUTPUT = "stage_06b_history_context"
STAGE_6C_OUTPUT = "stage_06c_icd_mapping"
STAGE_6E_OUTPUT = "stage_06e_lab_vital_codes"
STAGE_7_OUTPUT  = "stage_07_final_decision"

patients = sorted([p for p in RECORDS_DIR.iterdir() if p.is_dir() and p.name.startswith("patient_")])
print(f"Patients found: {len(patients)}")


Patients found: 15


## 2. Decision parameters -- ADJUST HERE

Defaults are the best-measured configuration from the table at the top. Every parameter is a
constant so the ablation and threshold sweep below can be re-run against any change upstream.

- `HISTORY_THRESHOLD = 0.3` -- keep prior codes at or above this 6b confidence. This is the
  single change that improves on the no-decision baseline. The optimum is a shallow plateau
  (0.2 -> 0.377, 0.3 -> 0.379, 0.4 -> 0.359), so 0.3 is a peak, not a cliff.
- `INCLUDE_SYMPTOM_CODES = True` -- whether 6c contributes candidates at all. Currently costs
  ~0.001 macro F1 (statistically indistinguishable) while adding recall, and keeps the
  architecture live so upstream improvements become visible. Set `False` to reproduce the
  best-measured number exactly.
- `MIN_SYMPTOM_SUPPORT = 2` -- 6c codes must be backed by at least this many inferred
  diagnoses. Admitting singly-supported codes drops macro F1 from 0.378 to 0.360.
- `SYMPTOM_WEIGHT`, `AGREEMENT_BONUS` -- scoring only; they affect ranking and the
  principal-diagnosis pick, not which codes pass the threshold. `AGREEMENT_BONUS` is kept
  deliberately small given the agreement finding above.


In [3]:
HISTORY_THRESHOLD     = 0.3
INCLUDE_SYMPTOM_CODES = True
MIN_SYMPTOM_SUPPORT   = 2

INCLUDE_LAB_CODES  = True   # Stage 6e -- codes derived from structured labs/vitals
MIN_LAB_CONFIDENCE = 0.0    # keep all 6e candidates by default; raise to admit only strong rules

SYMPTOM_WEIGHT  = 0.6   # how much a 6c mapping contributes to a candidate's score
LAB_WEIGHT      = 0.6   # how much a 6e lab/vital rule contributes
AGREEMENT_BONUS = 0.1   # small: agreement measured at 16.7% correct vs 29.9% for history alone


def load_history_codes(adm_dir: Path) -> dict:
    """{icd_code: {confidence, title, ...}} from Stage 6b."""
    p = adm_dir / STAGE_6B_OUTPUT / "history_codes.json"
    if not p.exists():
        return {}
    with open(p, encoding="utf-8") as f:
        return {e["icd_code"]: e for e in json.load(f)["prior_icd_codes"]}


def load_symptom_codes(adm_dir: Path) -> dict:
    """{icd_code: {confidence, n_supporting, ...}} from Stage 6c."""
    p = adm_dir / STAGE_6C_OUTPUT / "icd_candidates.json"
    if not p.exists():
        return {}
    with open(p, encoding="utf-8") as f:
        return {c["icd_code"]: c for c in json.load(f)["icd_candidates"]}


def load_lab_codes(adm_dir: Path) -> dict:
    """{icd_code: {confidence, evidence, tier, ...}} from Stage 6e."""
    p = adm_dir / STAGE_6E_OUTPUT / "lab_vital_codes.json"
    if not p.exists():
        return {}
    with open(p, encoding="utf-8") as f:
        return {c["icd_code"]: c for c in json.load(f)["icd_candidates"]}


def parse_ground_truth(adm_dir: Path):
    """(primary_code, {all codes}) from ground_truth.txt -- used ONLY for scoring."""
    p = adm_dir / "ground_truth.txt"
    primary, codes = None, set()
    if not p.exists():
        return primary, codes
    for line in p.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line.startswith("Primary ICD-10"):
            primary = line.split(":", 1)[-1].strip().replace(".", "").upper()
        m = re.match(r"^\s*\d+\.\s+([A-Z0-9]+)\s+\u2014", line)
        if m:
            codes.add(m.group(1).upper())
    return primary, codes


## 3. The decision

Each candidate carries its provenance so a reviewer can see *why* a code was predicted -- which
matters more here than the score itself, since a clinical coder auditing this needs the
justification, not a number.


In [4]:
def decide(history: dict, symptoms: dict, labs: dict = None) -> list:
    """Combine both sources into a thresholded, ranked final code list.

    A code qualifies if EITHER its history confidence clears HISTORY_THRESHOLD, or (when
    enabled) it is mapped from this visit's diagnoses with enough supporting inferences.
    Score drives ranking and the principal pick; the threshold decides membership."""
    candidates = {}

    for code, e in history.items():
        candidates[code] = {
            "code": code,
            "title": e.get("title", ""),
            "in_history": True,
            "history_confidence": e["confidence"],
            "history_recurrence": e.get("recurrence"),
            "in_symptoms": False,
            "symptom_support": 0,
            "symptom_route": None,
        }

    for code, c in symptoms.items():
        entry = candidates.setdefault(code, {
            "code": code, "title": c.get("title", ""), "in_history": False,
            "history_confidence": 0.0, "history_recurrence": None,
        })
        entry["in_symptoms"] = True
        entry["symptom_support"] = c.get("n_supporting", 1)
        entry["symptom_route"] = (c.get("routes") or [None])[0] if isinstance(c.get("routes"), list) else None
        if not entry.get("title"):
            entry["title"] = c.get("title", "")

    for code, c in labs.items():
        entry = candidates.setdefault(code, {
            "code": code, "title": c.get("title", ""), "in_history": False,
            "history_confidence": 0.0, "history_recurrence": None,
            "in_symptoms": False, "symptom_support": 0, "symptom_route": None,
        })
        entry["in_labs"] = True
        entry["lab_confidence"] = c.get("confidence", 0.0)
        entry["lab_evidence"] = c.get("evidence", "")
        entry["lab_tier"] = c.get("tier")
        if not entry.get("title"):
            entry["title"] = c.get("title", "")

    final = []
    for code, e in candidates.items():
        e.setdefault("in_labs", False)
        e.setdefault("lab_confidence", 0.0)
        e.setdefault("in_symptoms", False)
        e.setdefault("symptom_support", 0)

        keeps_history = e["in_history"] and e["history_confidence"] >= HISTORY_THRESHOLD
        keeps_symptom = (
            INCLUDE_SYMPTOM_CODES
            and e["in_symptoms"]
            and e["symptom_support"] >= MIN_SYMPTOM_SUPPORT
        )
        keeps_lab = (
            INCLUDE_LAB_CODES
            and e["in_labs"]
            and e["lab_confidence"] >= MIN_LAB_CONFIDENCE
        )
        if not (keeps_history or keeps_symptom or keeps_lab):
            continue

        score = e["history_confidence"]
        if e["in_symptoms"]:
            score += SYMPTOM_WEIGHT * min(1.0, e["symptom_support"] / 2.0)
        if e["in_labs"]:
            score += LAB_WEIGHT * e["lab_confidence"]
        if sum([e["in_history"], e["in_symptoms"], e["in_labs"]]) > 1:
            score += AGREEMENT_BONUS

        e["score"] = round(score, 4)
        srcs = ([("history" if e["in_history"] else None),
                 ("current_visit" if e["in_symptoms"] else None),
                 ("labs" if e["in_labs"] else None)])
        srcs = [s for s in srcs if s]
        e["source"] = "+".join(srcs)
        final.append(e)

    final.sort(key=lambda x: (-x["score"], x["code"]))
    for i, e in enumerate(final):
        e["rank"] = i + 1
        e["role"] = "principal" if i == 0 else "secondary"
    return final


def evaluate(final: list, gt_primary, gt_codes: set) -> dict:
    """Exact-match scoring. Category-level is reported alongside as a diagnostic -- it shows
    how much is lost to specificity rather than to being wrong outright."""
    pred = {e["code"] for e in final}
    tp = len(pred & gt_codes)
    precision = tp / len(pred) if pred else 0.0
    recall = tp / len(gt_codes) if gt_codes else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    pred_cats = {c[:3] for c in pred}
    cat_hits = len({g for g in gt_codes if g[:3] in pred_cats})

    return {
        "n_predicted": len(pred),
        "n_ground_truth": len(gt_codes),
        "n_correct": tp,
        "precision": round(precision, 3),
        "recall": round(recall, 3),
        "f1": round(f1, 3),
        "category_recall": round(cat_hits / len(gt_codes), 3) if gt_codes else 0.0,
        "principal_correct": bool(final) and final[0]["code"] == gt_primary,
        "principal_in_gt": bool(final) and final[0]["code"] in gt_codes,
    }


## 4. Run across all admissions

In [5]:
all_results = []

for patient_dir in patients:
    patient_id = patient_dir.name.replace("patient_", "")
    adm_root = patient_dir / "admissions"
    for adm_dir in sorted(adm_root.iterdir()) if adm_root.exists() else []:
        history  = load_history_codes(adm_dir)
        symptoms = load_symptom_codes(adm_dir)
        labs     = load_lab_codes(adm_dir)
        if not history and not symptoms and not labs:
            print(f"  SKIP {patient_id}/{adm_dir.name} -- no 6b or 6c output")
            continue

        gt_primary, gt_codes = parse_ground_truth(adm_dir)
        final = decide(history, symptoms, labs)
        ev = evaluate(final, gt_primary, gt_codes) if gt_codes else {}

        result = {
            "patient_id": patient_id,
            "admission_id": adm_dir.name.replace("hadm_", ""),
            "parameters": {
                "history_threshold": HISTORY_THRESHOLD,
                "include_symptom_codes": INCLUDE_SYMPTOM_CODES,
                "min_symptom_support": MIN_SYMPTOM_SUPPORT,
                "include_lab_codes": INCLUDE_LAB_CODES,
                "min_lab_confidence": MIN_LAB_CONFIDENCE,
            },
            "n_candidates_considered": len(set(history) | set(symptoms) | set(labs)),
            "final_codes": final,
            "evaluation": ev,
        }

        out_dir = adm_dir / STAGE_7_OUTPUT
        out_dir.mkdir(exist_ok=True)
        with open(out_dir / "final_icd_decision.json", "w", encoding="utf-8") as f:
            json.dump(result, f, indent=2)

        all_results.append(result)
        if ev:
            print(f'Patient {patient_id} | {adm_dir.name} | '
                  f'{ev["n_predicted"]:>3} predicted, {ev["n_correct"]:>2} correct '
                  f'| P={ev["precision"]:.3f} R={ev["recall"]:.3f} F1={ev["f1"]:.3f}')

print(f"\nDone. {len(all_results)} admissions decided.")


Patient 10361982 | hadm_24286431 |  11 predicted,  2 correct | P=0.182 R=0.400 F1=0.250
Patient 10426859 | hadm_29908281 |  24 predicted, 15 correct | P=0.625 R=0.682 F1=0.652
Patient 10458324 | hadm_21744342 |  12 predicted,  2 correct | P=0.167 R=0.167 F1=0.167
Patient 11251337 | hadm_29568708 |  11 predicted,  1 correct | P=0.091 R=0.143 F1=0.111
Patient 11474876 | hadm_29672491 |  24 predicted,  9 correct | P=0.375 R=0.529 F1=0.439
Patient 11607177 | hadm_23293838 |  23 predicted,  9 correct | P=0.391 R=0.692 F1=0.500
Patient 12007928 | hadm_23749816 |  30 predicted, 14 correct | P=0.467 R=0.737 F1=0.571
Patient 13196707 | hadm_21475988 |  34 predicted,  9 correct | P=0.265 R=0.281 F1=0.273
Patient 13508515 | hadm_21834271 |  16 predicted,  5 correct | P=0.312 R=0.357 F1=0.333
Patient 13952483 | hadm_23852410 |  47 predicted, 15 correct | P=0.319 R=0.600 F1=0.417
Patient 16014068 | hadm_29042843 |  29 predicted, 12 correct | P=0.414 R=0.632 F1=0.500
Patient 17774110 | hadm_27339772

## 5. Inspect one admission

In [6]:
EXAMPLE_IDX = 11
ex = all_results[EXAMPLE_IDX]

print(f'Patient {ex["patient_id"]} | Admission {ex["admission_id"]}')
print(f'{ex["n_candidates_considered"]} candidates considered -> {len(ex["final_codes"])} kept')
print(f'Evaluation: {ex["evaluation"]}')
print()
_, gt_codes = parse_ground_truth(
    RECORDS_DIR / f'patient_{ex["patient_id"]}' / "admissions" / f'hadm_{ex["admission_id"]}')
print(f'{"rank":<5}{"code":<9}{"score":>6}  {"source":<13}{"hit":<5}title')
print("-" * 88)
for e in ex["final_codes"][:20]:
    hit = "OK" if e["code"] in gt_codes else ""
    print(f'{e["rank"]:<5}{e["code"]:<9}{e["score"]:>6.2f}  {e["source"]:<13}{hit:<5}{e["title"][:38]}')


Patient 17774110 | Admission 27339772
39 candidates considered -> 29 kept
Evaluation: {'n_predicted': 29, 'n_ground_truth': 29, 'n_correct': 8, 'precision': 0.276, 'recall': 0.276, 'f1': 0.276, 'category_recall': 0.414, 'principal_correct': False, 'principal_in_gt': False}

rank code      score  source       hit  title
----------------------------------------------------------------------------------------
1    K7210      1.30  history+current_visit     Chronic hepatic failure without coma
2    F17210     0.90  history           Nicotine dependence, cigarettes, uncom
3    F319       0.90  history      OK   Bipolar disorder, unspecified
4    G4700      0.90  history           Insomnia, unspecified
5    J449       0.90  history      OK   Chronic obstructive pulmonary disease,
6    K219       0.90  history      OK   Gastro-esophageal reflux disease witho
7    K2270      0.90  history      OK   Barrett's esophagus without dysplasia
8    K7200      0.80  history+current_visit     Acute and 

## 6. Ablation -- what does each half actually contribute?

The number that matters as the pipeline evolves. Reports **both** macro F1 (mean of per-admission
F1, so every patient counts equally) and micro F1 (pooled over all codes, so every code counts
equally). They diverge when per-admission prediction counts are small, and reporting only one
invites the question of why that one.


In [7]:
def score_selection(selector) -> dict:
    """Run an arbitrary code-selection rule across all admissions, macro + micro."""
    macro_f1, micro_tp, micro_pred, micro_gt = [], 0, 0, 0
    for res in all_results:
        adm_dir = RECORDS_DIR / f'patient_{res["patient_id"]}' / "admissions" / f'hadm_{res["admission_id"]}'
        _, gt = parse_ground_truth(adm_dir)
        if not gt:
            continue
        pred = selector(load_history_codes(adm_dir), load_symptom_codes(adm_dir), load_lab_codes(adm_dir))
        tp = len(pred & gt)
        p = tp / len(pred) if pred else 0.0
        r = tp / len(gt)
        macro_f1.append(2 * p * r / (p + r) if (p + r) > 0 else 0.0)
        micro_tp += tp; micro_pred += len(pred); micro_gt += len(gt)

    mp = micro_tp / micro_pred if micro_pred else 0.0
    mr = micro_tp / micro_gt if micro_gt else 0.0
    return {
        "macro_f1": round(sum(macro_f1) / len(macro_f1), 3) if macro_f1 else 0.0,
        "micro_f1": round(2 * mp * mr / (mp + mr), 3) if (mp + mr) > 0 else 0.0,
        "precision": round(mp, 3),
        "recall": round(mr, 3),
        "n_predicted": micro_pred,
    }


ablation = {
    "history only (no threshold)":  lambda h, s, l: set(h),
    f"history >= {HISTORY_THRESHOLD}":
        lambda h, s, l: {c for c, e in h.items() if e["confidence"] >= HISTORY_THRESHOLD},
    "current visit only (6c)":      lambda h, s, l: set(s),
    "labs/vitals only (6e)":        lambda h, s, l: set(l),
    "6c from symptom clusters only":
        lambda h, s, l: {c for c, e in s.items() if "cluster" in (e.get("origins") or ["cluster"])},
    "6c from labs/imaging only (6d studies pass)":
        lambda h, s, l: {c for c, e in s.items() if "diagnostic_studies" in (e.get("origins") or [])},
    f"history >= {HISTORY_THRESHOLD} + labs":
        lambda h, s, l: {c for c, e in h.items() if e["confidence"] >= HISTORY_THRESHOLD} | set(l),
    "union of all three, no threshold":
        lambda h, s, l: set(h) | set(s) | set(l),
    "FINAL (this notebook's rule)":
        lambda h, s, l: {e["code"] for e in decide(h, s, l)},
}

rows = [{"rule": name, **score_selection(fn)} for name, fn in ablation.items()]
print(pd.DataFrame(rows).to_string(index=False))


                                       rule  macro_f1  micro_f1  precision  recall  n_predicted
                history only (no threshold)     0.363     0.367      0.297   0.482          401
                             history >= 0.3     0.379     0.396      0.354   0.449          314
                    current visit only (6c)     0.044     0.064      0.156   0.040           64
                      labs/vitals only (6e)     0.089     0.101      0.483   0.057           29
              6c from symptom clusters only     0.044     0.064      0.156   0.040           64
6c from labs/imaging only (6d studies pass)     0.000     0.000      0.000   0.000            0
                      history >= 0.3 + labs     0.389     0.405      0.353   0.474          331
           union of all three, no threshold     0.355     0.367      0.279   0.534          473
               FINAL (this notebook's rule)     0.389     0.405      0.353   0.474          331


## 7. Threshold sweep

Where the history threshold sits on the precision/recall curve, so the choice is visible rather
than asserted.


In [8]:
sweep = []
for t in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.9]:
    stats = score_selection(
        lambda h, s, l, t=t: (
            {c for c, e in h.items() if e["confidence"] >= t}
            | ({c for c, e in s.items() if e.get("n_supporting", 1) >= MIN_SYMPTOM_SUPPORT}
               if INCLUDE_SYMPTOM_CODES else set())
            | ({c for c, e in l.items() if e.get("confidence", 0) >= MIN_LAB_CONFIDENCE}
               if INCLUDE_LAB_CODES else set())
        )
    )
    sweep.append({"threshold": t, **stats})

df_sweep = pd.DataFrame(sweep)
print(df_sweep.to_string(index=False))
best = df_sweep.loc[df_sweep["macro_f1"].idxmax()]
print(f'\nBest macro F1 at threshold {best["threshold"]}: {best["macro_f1"]} '
      f'(currently set to {HISTORY_THRESHOLD})')


 threshold  macro_f1  micro_f1  precision  recall  n_predicted
       0.0     0.374     0.379      0.303   0.506          413
       0.1     0.381     0.389      0.319   0.498          385
       0.2     0.387     0.403      0.349   0.478          338
       0.3     0.389     0.405      0.353   0.474          331
       0.4     0.370     0.380      0.354   0.409          285
       0.5     0.365     0.374      0.364   0.385          261
       0.6     0.354     0.366      0.359   0.372          256
       0.7     0.343     0.356      0.416   0.312          185
       0.9     0.326     0.342      0.423   0.287          168

Best macro F1 at threshold 0.3: 0.389 (currently set to 0.3)


## 8. Save summary

In [9]:
principal_correct = sum(1 for r in all_results if r["evaluation"].get("principal_correct"))
principal_in_gt   = sum(1 for r in all_results if r["evaluation"].get("principal_in_gt"))
scored            = [r for r in all_results if r["evaluation"]]

summary = {
    "stage": "stage_07_final_icd_decision",
    "n_admissions": len(scored),
    "parameters": {
        "history_threshold": HISTORY_THRESHOLD,
        "include_symptom_codes": INCLUDE_SYMPTOM_CODES,
        "min_symptom_support": MIN_SYMPTOM_SUPPORT,
        "include_lab_codes": INCLUDE_LAB_CODES,
        "min_lab_confidence": MIN_LAB_CONFIDENCE,
        "symptom_weight": SYMPTOM_WEIGHT,
        "lab_weight": LAB_WEIGHT,
        "agreement_bonus": AGREEMENT_BONUS,
    },
    "macro_f1": round(sum(r["evaluation"]["f1"] for r in scored) / len(scored), 3),
    "mean_precision": round(sum(r["evaluation"]["precision"] for r in scored) / len(scored), 3),
    "mean_recall": round(sum(r["evaluation"]["recall"] for r in scored) / len(scored), 3),
    "mean_category_recall": round(sum(r["evaluation"]["category_recall"] for r in scored) / len(scored), 3),
    "principal_exact_correct": principal_correct,
    "principal_in_ground_truth": principal_in_gt,
    "ablation": rows,
    "threshold_sweep": sweep,
    "per_admission": [
        {"patient_id": r["patient_id"], "admission_id": r["admission_id"], **r["evaluation"]}
        for r in scored
    ],
}

out_path = RECORDS_DIR / "stage_07_summary.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(f"Saved: {out_path}")
print()
print(f'Macro F1              : {summary["macro_f1"]}')
print(f'Mean precision        : {summary["mean_precision"]}')
print(f'Mean recall           : {summary["mean_recall"]}')
print(f'Mean category recall  : {summary["mean_category_recall"]}  (exact-match ceiling diagnostic)')
print(f'Principal dx exactly right : {principal_correct}/{len(scored)}')
print(f'Principal dx in GT anywhere: {principal_in_gt}/{len(scored)}')


Saved: c:\Users\esnam\OneDrive\Desktop\esna_master_proj\ai-agents-for-clinical-coding\patient_records\stage_07_summary.json

Macro F1              : 0.389
Mean precision        : 0.34
Mean recall           : 0.468
Mean category recall  : 0.599  (exact-match ceiling diagnostic)
Principal dx exactly right : 0/15
Principal dx in GT anywhere: 10/15


## Where this leaves the pipeline

**Result**: macro F1 ~0.38, driven almost entirely by thresholded patient history. That is
roughly 13x the 0.028 baseline recorded for the old Stage 5 chain, but it is not a symptom-
understanding result -- it is a "chronic conditions recur" result.

**The honest position for the write-up**: the ontology work (Stages 5, 6, 6d, 6c) is
methodologically sound and each piece was validated in isolation, but on this cohort it does not
yet improve final code prediction over reading the patient's coding history. The ablation table
above is the evidence for that claim, and the mechanism for detecting when it changes.

**The three things most likely to change it**, in order of expected value:

1. **Coverage of chronic and status codes.** 17.4% of ground truth is chapter Z and much of the
   rest is chronic comorbidity. These are documented in the note's problem list and past medical
   history, which the pipeline currently discards -- Stage 5 routes only `CURRENT_SYMPTOMS`.
   `DOCUMENTED_DIAGNOSES` is carried forward untouched and never mapped to codes. Mapping that
   branch through 6c is the single largest untapped source, and it needs no new modelling.
2. **Specificity.** Category recall running at roughly double exact recall says a large share of
   predictions are in the right family and lose on the last digit. Using the map's `mapAdvice`
   (which often names the additional detail required) plus labs already sitting in
   `structured_labs.json` could recover some.
3. **Scale.** 15 admissions and 247 codes make every one of these numbers fragile -- the
   agreement finding rests on 6 overlapping codes. Nothing here should be treated as settled
   until it is re-run on a larger cohort.
